# 0.0. Set up

In [1]:
import os
from pathlib import Path

import re
import shutil
import pickle
import numpy as np
import pandas as pd
import openpyxl as opxl
import matplotlib.pyplot as plt
from typing import Optional, Dict, Any, List

from difflib import SequenceMatcher
from sklearn.impute import SimpleImputer, KNNImputer

# None means 'no limit'
pd.set_option('display.max_columns', None)

PROJECT_DIR = Path.cwd().parent

DATA_DIR = PROJECT_DIR / 'data'
DATA_PUBLIC_DIR = PROJECT_DIR / 'data' / 'public'
DATA_PRIVATE_DIR = PROJECT_DIR / 'data' / 'private'
DATA_PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'
OUTPUT_DIR = PROJECT_DIR / 'output'

# 1.0. Locate files

In [2]:
!ls '{DATA_PRIVATE_DIR}/ESC'

'SY 2022-2023'	'SY 2023-2024'	'SY 2024-2025'


In [3]:
dir_esc = os.path.join(DATA_PRIVATE_DIR, 'ESC')
folder_paths = [os.path.join(dir_esc, name) for name in os.listdir(dir_esc)]
display(folder_paths)

['/workspace/project_paaral/data/private/ESC/SY 2022-2023',
 '/workspace/project_paaral/data/private/ESC/SY 2023-2024',
 '/workspace/project_paaral/data/private/ESC/SY 2024-2025']

## Important note on LRN Formatting and Discrepancies 

Sept 13, 2025 - From DepEd ICTS

Non-standard LRNs: LRNs that do not conform to the standard 12-digit format are present in the data. This is attributed to:
BRAC Learning Centers (LCs): Some learners were initially enrolled in a BRAC LC, which used a 5-digit school code, resulting in a different LRN structure.
Some LRNs have discrepancies because they were issued during the first phase of the LIS implementation.

1. Missing LRNs: Some learners do not have an LRN listed. To address this, we need the learner's basic profile (First Name, Middle Name, Last Name, and Date of Birth) to search for their existing records in the system.
2. LRNs starting with '7': These LRNs were issued to learners by our Philippine Schools Overseas.
3. LRNs starting with '9': These LRNs belong to learners from State Universities and Colleges (SUCs) or Local Universities and Colleges (LUCs) that previously used a school ID starting with '9'.

# 2.0. Load data

## 2.1. One sheet

In [4]:
!ls '{DATA_PRIVATE_DIR}/ESC/SY 2024-2025'

BARMM.xlsx  R01.xlsx  R04-A.xlsx  R06.xlsx  R09.xlsx  R12.xlsx
CAR.xlsx    R02.xlsx  R04-B.xlsx  R07.xlsx  R10.xlsx  R13.xlsx
NCR.xlsx    R03.xlsx  R05.xlsx	  R08.xlsx  R11.xlsx


In [5]:
%%time
# Load the workbook
fpath = DATA_PRIVATE_DIR / 'ESC/SY 2024-2025/R04-A.xlsx'
workbook = opxl.load_workbook(fpath, read_only=True)
sheets = workbook.sheetnames
print(sheets)

['R04-A']
CPU times: user 5.49 s, sys: 77.4 ms, total: 5.57 s
Wall time: 6.09 s


In [6]:
%%time
# Is engine='calamine' faster? Yes.
xl = pd.ExcelFile(fpath, engine='calamine')
display(xl.sheet_names)

['R04-A']

CPU times: user 54.4 ms, sys: 52.2 ms, total: 107 ms
Wall time: 158 ms


In [7]:
%%time
df_region = pd.read_excel(
    fpath,
    sheet_name=sheets[0],
    usecols=[
        'DepEd ID',
        'LRN',
        'Full Name',
        'School ID',
        'School Name',
        'Grade',
        'Billing Statement No.',
        'Amount'
    ],
    dtype={
        'DepEd ID': str,  # Keep as string to preserve leading zeros
        'LRN': str,  # Keep as string to preserve leading zeros
        'Amount': int,
    },
    engine='calamine' # For faster reading
)
print(df_region.shape)

(163908, 8)
CPU times: user 1.68 s, sys: 225 ms, total: 1.9 s
Wall time: 1.94 s


In [8]:
tmp_df = df_region.copy()

tmp_df.columns = ['_'.join(col.strip().lower().split(' ')) for col in tmp_df.columns]

col_enc = {
    'deped_id':'school_id',
    'school_id':'esc_school_id',
    'grade':'grade_level',
    'billing_statement_no.':'billing_statement_no',
    'amount':'esc_subsidy_amount'
}
tmp_df.rename(columns=col_enc, inplace=True)

# Conform grade_level with enrollment data grade_level
tmp_df['grade_level'] = tmp_df['grade_level'].apply(lambda x: 'grade_' + str(x).strip())

print(tmp_df.shape)
display(tmp_df.head())

(163908, 8)


,school_id,lrn,full_name,esc_school_id,school_name,grade_level,billing_statement_no,esc_subsidy_amount
0,402944,109350170515,JANE MARIEL ENGRATO ABIERTA,402698,Rockfort Christian Academy,grade_7,ESC-265908,9000
1,402944,109329160351,Robielyn Ribon Adan,402698,Rockfort Christian Academy,grade_8,ESC-265908,9000
2,402944,402944170028,Ansleigh Xian Flores Adaro,402698,Rockfort Christian Academy,grade_8,ESC-265908,9000
3,402944,425701160003,PRINCE ANGELO CASTILLO AGUSTIN,402698,Rockfort Christian Academy,grade_7,ESC-265908,9000
4,402944,109329160412,Princess Rhea Asuncion Akagi,402698,Rockfort Christian Academy,grade_10,ESC-265908,9000


### Function

In [9]:
def process_one_sheet(fpath: str, sheet_name: str) -> pd.DataFrame:
    df_sheet = pd.read_excel(
        fpath,
        sheet_name=sheet_name,
        usecols=[
            'DepEd ID',
            'LRN',
            'Full Name',
            'School ID',
            'School Name',
            'Grade',
            'Billing Statement No.',
            'Amount'
        ],
        dtype={
            'DepEd ID': str,  # Keep as string to preserve leading zeros
            'LRN': str,  # Keep as string to preserve leading zeros
            'Amount': int,
        },
        engine='calamine' # For faster reading
    )
    
    df_sheet.columns = ['_'.join(col.strip().lower().split(' ')) for col in df_sheet.columns]
    
    col_enc = {
        'deped_id':'school_id',
        'school_id':'esc_school_id',
        'grade':'grade_level',
        'billing_statement_no.':'billing_statement_no',
        'amount':'esc_subsidy_amount'
    }
    df_sheet.rename(columns=col_enc, inplace=True)
    
    # Conform grade_level with enrollment data grade_level
    df_sheet['grade_level'] = df_sheet['grade_level'].apply(lambda x: 'grade_' + str(x).strip())

    # Tag the sheet name for tracking
    df_sheet['source_sheet_name'] = sheet_name

    return df_sheet

## 2.2. One Excel

### Function

In [10]:
def process_workbook(fpath: str, school_year: Optional[str] = None) -> pd.DataFrame:
    print(f'Processing file {fpath}.')
    sy = re.findall(r'SY (.*)/', fpath)[0] if school_year == None else school_year
    
    xl = pd.ExcelFile(fpath, engine='calamine')
    # In case an Excel file has multiple sheets
    sheets = xl.sheet_names 

    dfs = []
    for sheet in sheets:
        df_sheet = process_one_sheet(fpath, sheet)
        dfs.append(df_sheet)

    dfs = pd.concat(dfs)

    # Tag school year for tracking
    dfs['school_year'] = sy
    dfs['school_year'] = dfs['school_year'].astype(str)

    return dfs

In [11]:
fpath = str(DATA_PRIVATE_DIR / 'ESC/SY 2024-2025/R04-A.xlsx')
df_4a = process_workbook(fpath)
print(df_4a.shape)
display(df_4a.head(3))

Processing file /workspace/project_paaral/data/private/ESC/SY 2024-2025/R04-A.xlsx.
(163908, 10)


,school_id,lrn,full_name,esc_school_id,school_name,grade_level,billing_statement_no,esc_subsidy_amount,source_sheet_name,school_year
0,402944,109350170515,JANE MARIEL ENGRATO ABIERTA,402698,Rockfort Christian Academy,grade_7,ESC-265908,9000,R04-A,2024-2025
1,402944,109329160351,Robielyn Ribon Adan,402698,Rockfort Christian Academy,grade_8,ESC-265908,9000,R04-A,2024-2025
2,402944,402944170028,Ansleigh Xian Flores Adaro,402698,Rockfort Christian Academy,grade_8,ESC-265908,9000,R04-A,2024-2025


## 2.3. One SY

### Function

In [12]:
def process_school_year(dirpath: str) -> pd.DataFrame:
    school_year = re.findall(r'SY (.*)', dirpath)[0]
    
    sy_fnames = os.listdir(dirpath)
    sy_fpaths = [os.path.join(dirpath, fname) for fname in sy_fnames]

    dfs_sy = []
    for fpath in sy_fpaths:
        df_wk = process_workbook(fpath, school_year)
        dfs_sy.append(df_wk)

    dfs_sy = pd.concat(dfs_sy, ignore_index=True)

    # source_sheet_name is essentially 'region'. We conform this to regular
    # Deped region-naming. Thanks LLM for mapping it for me <3
    region_mapping = {
        'R01': 'Region I',
        'R02': 'Region II',
        'R03': 'Region III',
        'R04-A': 'Region IV-A',
        'R04-B': 'MIMAROPA',
        'R05': 'Region V',
        'R06': 'Region VI',
        'R07': 'Region VII',
        'R08': 'Region VIII',
        'R09': 'Region IX',
        'R10': 'Region X',
        'R11': 'Region XI',
        'R12': 'Region XII',
        'R13': 'CARAGA',
        'BARMM': 'BARMM',
        'CAR': 'CAR',
        'NCR': 'NCR'
    }

    dfs_sy['old_region'] = dfs_sy['source_sheet_name'].map(region_mapping)
    dfs_sy['old_region'] = dfs_sy['old_region'].astype(str)

    # Rearrange columns
    order_cols = ['old_region'] + dfs_sy.loc[:,:'school_year'].columns.tolist()
    dfs_sy = dfs_sy[order_cols]

    return dfs_sy

In [14]:
dirpath = str(DATA_PRIVATE_DIR / 'ESC/SY 2024-2025')
dfs_sy = process_school_year(dirpath)
print(dfs_sy.shape)

Processing file /workspace/project_paaral/data/private/ESC/SY 2024-2025/BARMM.xlsx.
Processing file /workspace/project_paaral/data/private/ESC/SY 2024-2025/CAR.xlsx.
Processing file /workspace/project_paaral/data/private/ESC/SY 2024-2025/NCR.xlsx.
Processing file /workspace/project_paaral/data/private/ESC/SY 2024-2025/R01.xlsx.
Processing file /workspace/project_paaral/data/private/ESC/SY 2024-2025/R02.xlsx.
Processing file /workspace/project_paaral/data/private/ESC/SY 2024-2025/R03.xlsx.
Processing file /workspace/project_paaral/data/private/ESC/SY 2024-2025/R04-A.xlsx.
Processing file /workspace/project_paaral/data/private/ESC/SY 2024-2025/R04-B.xlsx.
Processing file /workspace/project_paaral/data/private/ESC/SY 2024-2025/R05.xlsx.
Processing file /workspace/project_paaral/data/private/ESC/SY 2024-2025/R06.xlsx.
Processing file /workspace/project_paaral/data/private/ESC/SY 2024-2025/R07.xlsx.
Processing file /workspace/project_paaral/data/private/ESC/SY 2024-2025/R08.xlsx.
Processing

In [15]:
display(dfs_sy.head())

,old_region,school_id,lrn,full_name,esc_school_id,school_name,grade_level,billing_statement_no,esc_subsidy_amount,source_sheet_name,school_year
0,BARMM,405797,131248170058,SHIEHA MEA PASANDALAN ABAS,1201705,"Antonio R. Pacheco College, Inc.",grade_7,ESC-264510,9000,BARMM,2024-2025
1,BARMM,405797,469516170007,SITTIE FATHMA SHEMRAN LAMO ABAS,1201705,"Antonio R. Pacheco College, Inc.",grade_7,ESC-264510,9000,BARMM,2024-2025
2,BARMM,405797,405604150009,Aneeqa Farrah Abas Abas,1201705,"Antonio R. Pacheco College, Inc.",grade_8,ESC-264510,9000,BARMM,2024-2025
3,BARMM,405797,131248160083,Aisa Mae Pasandalan Abas,1201705,"Antonio R. Pacheco College, Inc.",grade_8,ESC-264510,9000,BARMM,2024-2025
4,BARMM,405797,131240170832,PRINCESS LYKA GEDTAL ABBAS,1201705,"Antonio R. Pacheco College, Inc.",grade_7,ESC-264510,9000,BARMM,2024-2025


In [16]:
dfs_sy['old_region'].unique()

array(['BARMM', 'CAR', 'NCR', 'Region I', 'Region II', 'Region III',
       'Region IV-A', 'MIMAROPA', 'Region V', 'Region VI', 'Region VII',
       'Region VIII', 'Region IX', 'Region X', 'Region XI', 'Region XII',
       'CARAGA'], dtype=object)

## 2.4. All

### Class

In [17]:
class ESCBeneficiaries:
    def __init__(
        self,
        dirpath_esc: Optional[str] = None,
        dirpath_school_info: Optional[str] = None,
        save_output: bool = False,
        verbose: bool = False
    ):
        # Paths
        self.dirpath_esc = str(DATA_PRIVATE_DIR / 'ESC') if dirpath_esc == None else dirpath_esc
        fpath_schinfo = 'processed_project_bukas_school_information.parquet'

        if not os.path.exists(OUTPUT_DIR):
            os.mkdir(OUTPUT_DIR)
        
        self.dirpath_school_info = (
            str(OUTPUT_DIR / fpath_schinfo) if dirpath_school_info == None else dirpath_school_info
        )
        self.fpath_save = str(OUTPUT_DIR / 'processed_esc_beneficiaries.parquet')

        # Storage
        self.esc_beneficiaries = None
        self.school_info = None
        self.checkpoint_df = None

        # Logging
        self.save_output = save_output
        self.verbose = verbose

    def process_one_sheet(
        self,
        fpath: str,
        sheet_name: str
    ) -> pd.DataFrame:
        df_sheet = pd.read_excel(
            fpath,
            sheet_name=sheet_name,
            usecols=[
                'DepEd ID',
                'LRN',
                'Full Name',
                'School ID',
                'School Name',
                'Grade',
                'Billing Statement No.',
                'Amount'
            ],
            dtype={
                'DepEd ID': str,  # Keep as string to preserve leading zeros
                'LRN': str,  # Keep as string to preserve leading zeros
                'Amount': int,
            },
            engine='calamine' # For faster reading
        )
        
        df_sheet.columns = ['_'.join(col.strip().lower().split(' ')) for col in df_sheet.columns]
        
        col_enc = {
            'deped_id':'destination_school_id',
            'school_id':'esc_school_id',
            'grade':'grade_level',
            'billing_statement_no.':'billing_statement_no',
            'amount':'esc_subsidy_amount'
        }
        df_sheet.rename(columns=col_enc, inplace=True)
        
        # Conform grade_level with enrollment data grade_level
        df_sheet['grade_level'] = df_sheet['grade_level'].apply(lambda x: 'grade_' + str(x).strip())
    
        # Tag the sheet name for tracking
        df_sheet['source_sheet_name'] = sheet_name
    
        return df_sheet

    def process_workbook(
        self,
        fpath: str,
        school_year: Optional[str] = None
    ) -> pd.DataFrame:
        # print(f'Processing file {fpath}.')
        sy = re.findall(r'SY (.*)/', fpath)[0] if school_year == None else school_year
        
        xl = pd.ExcelFile(fpath, engine='calamine')
        # In case an Excel file has multiple sheets
        sheets = xl.sheet_names 
    
        dfs = []
        for sheet in sheets:
            df_sheet = self.process_one_sheet(fpath, sheet)
            dfs.append(df_sheet)
    
        dfs = pd.concat(dfs)
    
        # Tag school year for tracking
        dfs['school_year'] = sy
        dfs['school_year'] = dfs['school_year'].astype(str)
    
        return dfs

    def process_school_year(
        self,
        dirpath: str
    ) -> pd.DataFrame:
        school_year = re.findall(r'SY (.*)', dirpath)[0]
        
        sy_fnames = os.listdir(dirpath)
        sy_fpaths = [os.path.join(dirpath, fname) for fname in sy_fnames]

        self._vprint(f"Processing {len(sy_fpaths)} Excel files in SY {school_year}...")
    
        dfs_sy = []
        for fpath in sy_fpaths:
            df_wk = self.process_workbook(fpath, school_year)
            dfs_sy.append(df_wk)
    
        dfs_sy = pd.concat(dfs_sy, ignore_index=True)
    
        # source_sheet_name is essentially 'region'. We conform this to regular
        # Deped region-naming. Thanks LLM for mapping it for me <3
        region_mapping = {
            'R01': 'Region I',
            'R02': 'Region II',
            'R03': 'Region III',
            'R04-A': 'Region IV-A',
            'R04-B': 'MIMAROPA',
            'R05': 'Region V',
            'R06': 'Region VI',
            'R07': 'Region VII',
            'R08': 'Region VIII',
            'R09': 'Region IX',
            'R10': 'Region X',
            'R11': 'Region XI',
            'R12': 'Region XII',
            'R13': 'CARAGA',
            'BARMM': 'BARMM',
            'CAR': 'CAR',
            'NCR': 'NCR'
        }
    
        dfs_sy['old_region'] = dfs_sy['source_sheet_name'].map(region_mapping)
        dfs_sy['old_region'] = dfs_sy['old_region'].astype(str)
    
        # Move 'old_region' to the front without dropping other columns
        cols = dfs_sy.columns.tolist()
        if 'old_region' in cols:
            cols.insert(0, cols.pop(cols.index('old_region')))
        dfs_sy = dfs_sy[cols]

        self._vprint(f"Shape of dataframe for SY {school_year} is {dfs_sy.shape}\n")
    
        return dfs_sy

    def _label_issued_lrns(self, df: pd.DataFrame) -> pd.DataFrame: 
        self._vprint(f"Labeling LRNs based on ICTs notes...")
        
        # Initialize the column with a default value to ensure it exists
        df['lrn_category'] = 'Unknown'
    
        ## 1. Handle NaNs
        mask_na = df['lrn'].isna()
        df.loc[mask_na, 'lrn_category'] = 'Missing'
    
        ## 2. Tag standard LRNs (12 digits)
        # Ensure we only check non-null values to avoid errors
        mask_standard = df['lrn'].fillna('').apply(lambda x: len(str(x)) == 12)
        df.loc[mask_standard & ~mask_na, 'lrn_category'] = 'Standard'
    
        ## 3. Tag non-standard LRNs (less than 12 digits)
        mask_non_standard = df['lrn'].fillna('').apply(lambda x: len(str(x)) < 12)
        df.loc[mask_non_standard & ~mask_na, 'lrn_category'] = 'Non-standard'
        
        ## 4. Tag PSO (Starts with 7)
        mask_pso = df['lrn'].astype(str).str.startswith('7')
        df.loc[mask_pso, 'lrn_category'] = 'PSO_issued'
    
        ## 5. Tag SUC/LUC (Starts with 9)
        mask_ucs = df['lrn'].astype(str).str.startswith('9')
        df.loc[mask_ucs, 'lrn_category'] = 'SUC_LUC_issued'
    
        return df

    def extract_school_id_from_lrn(self, bdf: pd.DataFrame) -> pd.DataFrame:
        df = self._label_issued_lrns(bdf)

        # Check if user has the school information parquet file
        if os.path.exists(self.dirpath_school_info):
            sch_info = pd.read_parquet(self.dirpath_school_info)
        else:
            print(f"Obtain the processed DepEd school information .parquet file")
            return False
        
        # Extract indiscriminately the first 6 digits of every LRN
        df['origin_school_id'] = df['lrn'].apply(lambda x: str(x)[:6])

        # Then, validate if extracted school ID exists in our school information table
        sch_info_ids = sch_info['school_id'].unique()
        mask = df['origin_school_id'].isin(sch_info_ids)
        df.loc[mask, 'is_valid_origin_school_id'] = True
        df.loc[~mask, 'is_valid_origin_school_id'] = False

        self._vprint(
            f"The count of VALID origin school IDs extracted from LRNs is: {df['is_valid_origin_school_id'].sum():,} of {df.shape[0]:,}"
        )

        return df
    
    def process(self):
        school_years = os.listdir(self.dirpath_esc)
        dirpaths_sy = [os.path.join(self.dirpath_esc, sy) for sy in school_years]

        dfs = []
        for dirpath in dirpaths_sy:
            df_sy = self.process_school_year(dirpath)
            dfs.append(df_sy)

        dfs = pd.concat(dfs, ignore_index=True)
        # self.checkpoint_df = dfs  # For debugging

        # Create column containing the most likely origin schools of these beneficiaries
        dfs = self.extract_school_id_from_lrn(dfs)
        
        self.esc_beneficiaries = dfs

        if self.save_output:
            self.save()
        
        return dfs

    def save(self):
        self._vprint(f"\nSaving processed ESC beneficiaries table to {self.fpath_save}")

        if os.path.exists(self.fpath_save):
            if os.path.isdir(self.fpath_save):
                shutil.rmtree(self.fpath_save)
                print(f"Removed directory with path {self.fpath_save}")
            else:
                os.remove(self.fpath_save)
                print(f"Removed file with path {self.fpath_save}")

        self.esc_beneficiaries.to_parquet(self.fpath_save, partition_cols='school_year')
        self._vprint(f"Saved!")

    def load_esc_data(self):
        self._vprint(f"Loading processed ESC beneficiaries table...")
        if os.path.exists(self.fpath_save):
            df = pd.read_parquet(self.fpath_save)
            self.esc_beneficiaries = df
            self._vprint(f"Loading finished!")
            
            return df
        else:
            print(f"Check if processed ESC parquet file exists.")

    def _vprint(self, msg: str):
        if self.verbose:
            print(msg)

### Implem

In [18]:
eb = ESCBeneficiaries(save_output=True, verbose=True)

In [19]:
df_benef = eb.process()

Processing 17 Excel files in SY 2022-2023...
Shape of dataframe for SY 2022-2023 is (920899, 11)

Processing 17 Excel files in SY 2023-2024...
Shape of dataframe for SY 2023-2024 is (897016, 11)

Processing 17 Excel files in SY 2024-2025...
Shape of dataframe for SY 2024-2025 is (863753, 11)

Labeling LRNs based on ICTs notes...
The count of VALID origin school IDs extracted from LRNs is: 2,516,924 of 2,681,668

Saving processed ESC beneficiaries table to /workspace/project_paaral/output/processed_esc_beneficiaries.parquet
Removed directory with path /workspace/project_paaral/output/processed_esc_beneficiaries.parquet
Saved!


In [20]:
print(df_benef.shape)
display(df_benef.head(3))
display(df_benef.tail(2))

(2681668, 14)


,old_region,destination_school_id,lrn,full_name,esc_school_id,school_name,grade_level,billing_statement_no,esc_subsidy_amount,source_sheet_name,school_year,lrn_category,origin_school_id,is_valid_origin_school_id
0,BARMM,475511,133526130177,Mae Ann Canoy Abajo,1603520,"Adiong Memorial College Foundation, Inc.",grade_9,ESC-227153,9000,BARMM,2022-2023,Standard,133526,True
1,BARMM,475511,133536150012,MA. JOY PAGUNTALAN ABAYA,1603520,"Adiong Memorial College Foundation, Inc.",grade_7,ESC-227153,9000,BARMM,2022-2023,Standard,133536,True
2,BARMM,475511,133536150011,MA. GRACE PAGUNTALAN ABAYA,1603520,"Adiong Memorial College Foundation, Inc.",grade_7,ESC-227153,9000,BARMM,2022-2023,Standard,133536,True


,old_region,destination_school_id,lrn,full_name,esc_school_id,school_name,grade_level,billing_statement_no,esc_subsidy_amount,source_sheet_name,school_year,lrn_category,origin_school_id,is_valid_origin_school_id
2681666,CARAGA,406039,115224140076,QUENNIE ALANDRIA TABARANGAO,1301787,Southern Technological Institute of the Philip...,grade_7,ESC-282698,9000,R13,2024-2025,Standard,115224,True
2681667,CARAGA,406039,132835140095,LISTER DAVE MANGLICMOT TUSAY,1301787,Southern Technological Institute of the Philip...,grade_8,ESC-282698,9000,R13,2024-2025,Standard,132835,True


In [21]:
df_benef['lrn_category'].value_counts()

lrn_category
Standard          2675623
PSO_issued           3983
Non-standard         1993
SUC_LUC_issued         62
Missing                 6
Unknown                 1
Name: count, dtype: int64

### Load

In [22]:
df_bf = eb.load_esc_data()
print(df_bf.shape)

Loading processed ESC beneficiaries table...
Loading finished!
(2681668, 14)


In [23]:
display(df_bf.head())

,old_region,destination_school_id,lrn,full_name,esc_school_id,school_name,grade_level,billing_statement_no,esc_subsidy_amount,source_sheet_name,lrn_category,origin_school_id,is_valid_origin_school_id,school_year
0,BARMM,475511,133526130177,Mae Ann Canoy Abajo,1603520,"Adiong Memorial College Foundation, Inc.",grade_9,ESC-227153,9000,BARMM,Standard,133526,True,2022-2023
1,BARMM,475511,133536150012,MA. JOY PAGUNTALAN ABAYA,1603520,"Adiong Memorial College Foundation, Inc.",grade_7,ESC-227153,9000,BARMM,Standard,133536,True,2022-2023
2,BARMM,475511,133536150011,MA. GRACE PAGUNTALAN ABAYA,1603520,"Adiong Memorial College Foundation, Inc.",grade_7,ESC-227153,9000,BARMM,Standard,133536,True,2022-2023
3,BARMM,475511,133519130013,Wenzy James Malones Abaya,1603520,"Adiong Memorial College Foundation, Inc.",grade_9,ESC-227153,9000,BARMM,Standard,133519,True,2022-2023
4,BARMM,475511,133519110001,ANGIELYN ACOSTA ABAYA,1603520,"Adiong Memorial College Foundation, Inc.",grade_10,ESC-227153,9000,BARMM,Standard,133519,True,2022-2023


In [24]:
# Check the count of Grade 7 beneficiaries in SY 23-24 in NCR, Region IV-A, and Region III
mask = (
    (df_bf['old_region'].isin(['NCR','Region IV-A', 'Region III']))
    & (df_bf['grade_level'] == 'grade_7')
    & (df_bf['school_year'] == '2023-2024')
    # & (df_bf['is_valid_origin_school_id'] == True)
)
len_mask = mask.sum()
print(len_mask)

96572
